# Example 7 — Toeplitz Factorisation: Levinson vs. Generalised Schur

A 10 000 × 10 000 symmetric positive-definite Toeplitz matrix has **100 million**
entries, but because every diagonal is constant it is fully described by a single
vector of length **10 000** — its first row.

This notebook demonstrates two algorithms that exploit this structure to compute
the **inverse Cholesky factors** `l1` and `l2` without ever forming the full matrix:

| Algorithm | Complexity | Hector class |
|-----------|-----------|-------------|
| Durbin–Levinson | O(*n*²) | `Levinson` |
| Generalised Schur (GSA) | O(*n* log² *n*) | `Schur` |

The covariance matrix used here comes from the **GGM (Generalised Gauss–Markov)**
noise model with flicker-noise spectral index *d* = 0.5, the standard choice for
daily GNSS position time series.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import toeplitz

from hector._ggm import create_t_inner          # fast Cython GGM covariance
from hector.levinson import Levinson
from hector.schur import Schur
from hector.control import SingletonMeta

%matplotlib inline

## 1. Build the GGM covariance vector

The GGM covariance between epochs separated by lag *j* is:

$$t_j = \sigma^2\,\frac{\Gamma(j+d)\,{}_2F_1(d,\,j+d;\,j+1-d;\,(1-\phi)^2)}
           {\Gamma(j+1-d)\,\Gamma(d)\,\Gamma(1-d)}$$

with *d* = 0.5 (flicker noise) and 1 − *φ* = 6.9 × 10⁻⁶ (long but finite
correlation length, making the process stationary).

The vector `t` of length *n* is the **only input** needed — the full *n* × *n*
matrix is never formed.

In [ ]:
n   = 10_000
d   = 0.5           # flicker-noise spectral index
phi = 6.9e-6        # 1 - phi  (GGM corner parameter)

t = create_t_inner(n, d, phi)
print(f"Covariance vector length : {len(t)}")
print(f"t[0] (variance)          : {t[0]:.6f}")
print(f"t[1]                     : {t[1]:.6f}")
print(f"t[-1]                    : {t[-1]:.6f}")
print(f"Compression ratio        : {n**2 // n}× (matrix entries vs vector length)")

In [ ]:
lags = np.arange(n)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(lags[:200], t[:200], color='steelblue', lw=1.2)
axes[0].set_xlabel('Lag (days)')
axes[0].set_ylabel('Covariance')
axes[0].set_title('GGM covariance — first 200 lags')

axes[1].loglog(lags[1:], t[1:], color='steelblue', lw=1.2)
axes[1].set_xlabel('Lag (days)')
axes[1].set_ylabel('Covariance')
axes[1].set_title('GGM covariance — log–log (all lags)')

fig.tight_layout()
plt.show()

## 2. The inverse Cholesky factors `l1` and `l2`

Every positive-definite matrix **C** has a Cholesky factorisation **C** = **L** **L**ᵀ.
The whitening matrix **L**⁻¹ transforms correlated noise into white noise.

Because **C** is Toeplitz, **L**⁻¹ has a special structure captured by just two
vectors `l1` and `l2`.  Together they encode **C**⁻¹ via the
**Gohberg–Semencul formula**:

$$\mathbf{C}^{-1} = \frac{1}{\delta}\left(\mathbf{L}_1^\top\mathbf{L}_1
                                          - \mathbf{L}_2^\top\mathbf{L}_2\right)$$

where **L**₁ and **L**₂ are lower-triangular Toeplitz matrices built from `l1`
and `l2`, and *δ* is the final prediction-error variance.
This reduces storage from O(*n*²) to O(*n*) and matrix–vector products
from O(*n*²) to O(*n* log *n*).

Both `Levinson` and `Schur` return `(l1, l2, delta, ln_det_C)`.

## 3. Levinson and Schur at *n* = 10 000

In [ ]:
SingletonMeta.clear_all()
t0 = time.perf_counter()
l1_lev, l2_lev, delta_lev, lndet_lev = Levinson().compute(t)
time_lev = time.perf_counter() - t0

SingletonMeta.clear_all()
t0 = time.perf_counter()
l1_gsa, l2_gsa, delta_gsa, lndet_gsa = Schur().compute(t)
time_gsa = time.perf_counter() - t0

print(f"Levinson  : {time_lev*1e3:6.1f} ms")
print(f"Schur/GSA : {time_gsa*1e3:6.1f} ms")
print(f"Speedup   : {time_lev/time_gsa:.1f}×")
print()
print(f"max |l1_lev − l1_gsa|  : {np.max(np.abs(l1_lev - l1_gsa)):.2e}")
print(f"max |l2_lev − l2_gsa|  : {np.max(np.abs(l2_lev - l2_gsa)):.2e}")
print(f"|lndet_lev − lndet_gsa|: {abs(lndet_lev - lndet_gsa):.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(l1_lev[:200], color='steelblue', lw=1.0, label='Levinson')
axes[0].plot(l1_gsa[:200], color='tomato',    lw=0.8, ls='--', label='Schur/GSA')
axes[0].set_xlabel('Index')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('l1 — first 200 coefficients')
axes[0].legend()

axes[1].semilogy(np.abs(l1_lev[1:]), color='steelblue', lw=1.0, label='Levinson')
axes[1].semilogy(np.abs(l1_gsa[1:]), color='tomato',    lw=0.8, ls='--', label='Schur/GSA')
axes[1].set_xlabel('Index')
axes[1].set_ylabel('|l1| (log scale)')
axes[1].set_title('l1 — all coefficients (semi-log)')
axes[1].legend()

fig.suptitle('Inverse Cholesky whitening filter l1  (d = 0.5, n = 10 000)', y=1.02)
fig.tight_layout()
plt.show()

## 4. Small-*n* sanity check

For *n* = 100 we can form the full matrix and verify the algebra directly.

**Defining property of the Levinson predictor:** the forward predictor `l1`
satisfies **C**·`l1` = *δ*·**e**₀, where **e**₀ = [1, 0, …, 0]ᵀ.

**Gohberg–Semencul check:** reconstruct **C**⁻¹ from `l1`, `l2`, *δ* and
compare with NumPy's dense inverse.

In [ ]:
n_small  = 100
t_small  = create_t_inner(n_small, d, phi)
C_small  = toeplitz(t_small)

SingletonMeta.clear_all()
l1_s, l2_s, delta_s, _ = Levinson().compute(t_small)

# Property 1: C·l1 = delta·e0
e0 = np.zeros(n_small); e0[0] = delta_s
err_pred = np.max(np.abs(C_small @ l1_s - e0))
print(f"max |C·l1 − delta·e0|      : {err_pred:.2e}  (expect ≈ machine precision)")

# Property 2: Gohberg–Semencul formula
L1     = np.tril(toeplitz(l1_s))          # lower-triangular Toeplitz from l1
L2     = np.tril(toeplitz(l2_s))          # lower-triangular Toeplitz from l2
Cinv_gs = (L1.T @ L1 - L2.T @ L2) / delta_s
Cinv_np = np.linalg.inv(C_small)
err_gs  = np.max(np.abs(Cinv_gs - Cinv_np))
print(f"max |C⁻¹_GS − C⁻¹_numpy|  : {err_gs:.2e}  (expect ≈ machine precision)")

## 5. Timing over a range of *n*

Levinson grows as O(*n*²) and the GSA as O(*n* log² *n*), so the speedup
increases with series length.

In [ ]:
ns      = [500, 1000, 2000, 5000, 10000]
t_lev_v = []
t_gsa_v = []

for ni in ns:
    ti = create_t_inner(ni, d, phi)

    SingletonMeta.clear_all()
    t0 = time.perf_counter()
    Levinson().compute(ti)
    t_lev_v.append(time.perf_counter() - t0)

    SingletonMeta.clear_all()
    t0 = time.perf_counter()
    Schur().compute(ti)
    t_gsa_v.append(time.perf_counter() - t0)

    print(f"n={ni:6d}  Levinson={t_lev_v[-1]*1e3:7.1f} ms  "
          f"GSA={t_gsa_v[-1]*1e3:6.1f} ms  "
          f"speedup={t_lev_v[-1]/t_gsa_v[-1]:.1f}×")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].loglog(ns, [x*1e3 for x in t_lev_v], 'o-', color='steelblue', label='Levinson  O(n²)')
axes[0].loglog(ns, [x*1e3 for x in t_gsa_v], 's-', color='tomato',    label='Schur/GSA  O(n log²n)')
axes[0].set_xlabel('n')
axes[0].set_ylabel('Time (ms)')
axes[0].set_title('Factorisation time')
axes[0].legend()

speedups = [l/g for l, g in zip(t_lev_v, t_gsa_v)]
axes[1].plot(ns, speedups, 'D-', color='seagreen')
axes[1].axhline(1, color='gray', lw=0.8, ls='--')
axes[1].set_xlabel('n')
axes[1].set_ylabel('Speedup (×)')
axes[1].set_title('GSA speedup over Levinson')

fig.tight_layout()
plt.show()